# 05 · Avaliação final no PTB-XL v1.0.3

**DCA3606 — Inteligência Artificial · UFRN**

As duas tarefas são **Normal (0) × Alterado (1)** e **estimativa da idade cronológica em anos**. Em cada tarefa, a MLP, a CNN 1D, a ResNet 1D e os dois Random Forests recebem todos os ECGs elegíveis do fold 10. As referências constantes são definidas exclusivamente com os folds 1–8. Os grupos de exames podem diferir entre tarefas, pois os critérios de elegibilidade dos alvos são distintos.

No Colab, abra este notebook, selecione GPU se disponível e execute as células em ordem. Autorize a montagem do Drive. Dependências: Python, NumPy, pandas, PyTorch, scikit-learn, joblib, Matplotlib e IPython. Caso falte uma biblioteca, instale-a antes de executar. Para abrir os arquivos Joblib, use a **mesma versão do scikit-learn que serializou os modelos**. Use apenas artefatos produzidos pelo próprio projeto.

A raiz no Drive é `/content/drive/MyDrive/IA-2026`. São necessários os módulos `src/deep_learning.py`, `src/autoencoder_classicos.py` e `src/__init__.py`, os cinco arquivos da preparação e as duas pastas completas `treino_completo_v1`. O inventário abaixo mostra a lista exata. Cada execução gera uma pasta própria em `resultados/avaliacao_final/`. Os sinais preparados são lidos em lotes por memória mapeada. Por padrão, uma cópia temporária do binário no disco do Colab acelera a leitura.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from tempfile import TemporaryDirectory
import json
import os
import platform
import shutil
import sys
from time import perf_counter
import warnings

from google.colab import drive
from IPython.display import display, Markdown, Image
import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, mean_absolute_error,
                             mean_squared_error, precision_score, r2_score, recall_score)

drive.mount('/content/drive')
RAIZ = Path('/content/drive/MyDrive/IA-2026')
RESULTADOS = RAIZ / 'resultados'
DADOS = RESULTADOS / 'preprocessamento'
DEEP = RESULTADOS / 'deep_learning' / 'treino_completo_v1'
CLASSICOS = RESULTADOS / 'autoencoder_classicos' / 'treino_completo_v1'
BASE_SAIDA = RESULTADOS / 'avaliacao_final'
SEMENTE = 3606
LOTE = 64
REPETICOES = 3
COPIAR_BINARIO_PARA_CONTENT = True
DISPOSITIVO = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(SEMENTE)
torch.manual_seed(SEMENTE)
if DISPOSITIVO.type == 'cuda':
    torch.cuda.manual_seed_all(SEMENTE)

TAREFAS = ('classificacao', 'regressao')
ARQUITETURAS = ('mlp', 'cnn1d', 'resnet1d')
REPRESENTACOES = ('original', 'embedding')
ALVOS = {'classificacao': 'alvo_classificacao', 'regressao': 'alvo_idade'}
DIMENSAO_ORIGINAL = 12000
CLASSES = ['Normal', 'Alterado']
NOMES = dict(zip(
    (*ARQUITETURAS, 'rf_original', 'rf_embedding', 'baseline'),
    ('MLP', 'CNN 1D', 'ResNet 1D', 'RF · sinal original', 'RF · embedding', 'Baseline')
))

## 1. Inventário e auditoria dos dados

Os nomes abaixo vêm dos arquivos produzidos pelas etapas 2–4. O inventário verifica a presença e o tamanho no Drive antes de abrir qualquer modelo. A auditoria cruza os CSVs com o manifesto pelo `ecg_id`, confere os folds, o isolamento por paciente, a ordem e a unicidade de `linha_sinal` e os parâmetros da normalização ajustada nos folds 1–8.

In [ ]:
BINARIO = DADOS / 'sinais_normalizados.float32.bin'
PASTAS_DL = {(t, a): DEEP / t / a for t in TAREFAS for a in ARQUITETURAS}
AE_CHECKPOINT = CLASSICOS / 'checkpoints' / 'autoencoder_melhor.pt'
AE_CONFIG = CLASSICOS / 'configuracoes' / 'autoencoder.json'
AE_HISTORICO = CLASSICOS / 'metricas' / 'historico_autoencoder.csv'
AE_INDICE = CLASSICOS / 'embeddings' / 'indice_embeddings.csv'
AE_META = CLASSICOS / 'embeddings' / 'metadados_embeddings.json'
AE_Z_AJUSTE = CLASSICOS / 'embeddings' / 'representacoes.npy'
GRAFICOS_AE = [CLASSICOS / 'graficos' / nome for nome in
               ('perda_reconstrucao.png', 'exemplos_reconstrucao.png')]
RF_ARQUIVOS = {}
for tarefa in TAREFAS:
    for rep in REPRESENTACOES:
        prefixo = f'random_forest_{tarefa}_{rep}'
        RF_ARQUIVOS[tarefa, rep] = (
            CLASSICOS / 'modelos' / f'{prefixo}.joblib',
            CLASSICOS / 'configuracoes' / f'{prefixo}.json',
            CLASSICOS / 'metricas' / f'{prefixo}.json',
        )

requeridos = [RAIZ / 'src' / nome for nome in
              ('__init__.py', 'deep_learning.py', 'autoencoder_classicos.py')]
requeridos += [DADOS / nome for nome in
               ('normalizacao.json', 'manifesto.csv', 'classificacao.csv', 'regressao.csv')]
requeridos += [BINARIO, AE_CHECKPOINT, AE_CONFIG, AE_HISTORICO, AE_INDICE, AE_META,
               AE_Z_AJUSTE, *GRAFICOS_AE]
requeridos += [pasta / nome for pasta in PASTAS_DL.values() for nome in
               ('melhor.pt', 'configuracao.json', 'historico.csv', 'curvas.png')]
requeridos += [caminho for grupo in RF_ARQUIVOS.values() for caminho in grupo]
inventario = pd.DataFrame([
    {'arquivo': str(c.relative_to(RAIZ)), 'bytes': c.stat().st_size if c.is_file() else 0}
    for c in requeridos
])
display(inventario)
faltantes = inventario.loc[inventario.bytes.eq(0), 'arquivo']
if not faltantes.empty:
    raise FileNotFoundError('Artefatos ausentes ou vazios em ' + str(RAIZ) + '\n' +
                            '\n'.join(faltantes))

sys.dont_write_bytecode = True
sys.path.insert(0, str(RAIZ))
from src.deep_learning import criar_modelo
from src.autoencoder_classicos import (
    AutoencoderECG1D, conferir_correspondencia, selecionar_amostras,
    _assinatura_amostra, _sha256_arquivo,
)

In [ ]:
def exigir(condicao, descricao):
    if not condicao:
        raise ValueError(descricao)


def ler_json(caminho):
    return json.loads(caminho.read_text(encoding='utf-8'))


normalizacao, tabelas = conferir_correspondencia(DADOS)
shape = tuple(normalizacao['shape_sinais'])
manifesto = tabelas['manifesto']
cols_id = ['ecg_id', 'patient_id', 'strat_fold', 'particao', 'linha_sinal']
aprovados = manifesto.loc[manifesto.status_sinal.eq('aprovado')]
exigir(normalizacao['metodo'] == 'z_score_global_por_derivacao',
       'Método de normalização inesperado.')
media = np.asarray(normalizacao['media_treino'])
desvio = np.asarray(normalizacao['desvio_treino'])
exigir(media.shape == desvio.shape == (12,) and np.isfinite(media).all()
       and np.isfinite(desvio).all() and (desvio > 0).all(),
       'Parâmetros de normalização inválidos.')
exigir(normalizacao['derivacoes'] ==
       ['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6'],
       'Ordem das derivações incompatível com o treinamento.')
exigir(np.array_equal(np.sort(aprovados.linha_sinal.to_numpy()), np.arange(shape[0])),
       'As linhas dos sinais aprovados devem indexar a matriz sem lacunas ou duplicatas.')
exigir(normalizacao['registros_ajuste'] == int(aprovados.particao.eq('treino').sum()),
       'Contagem de sinais usados para ajustar a normalização divergente.')

testes, constantes_baseline, contagens = {}, {}, {}
for tarefa in TAREFAS:
    tabela = tabelas[tarefa]
    alvo = ALVOS[tarefa]
    ids_esperados = aprovados.loc[aprovados[alvo].notna(), 'ecg_id']
    exigir(set(tabela.ecg_id) == set(ids_esperados),
           f'{tarefa}: os ECGs diferem da lista de elegíveis aprovados no manifesto.')
    contagens[tarefa] = tabela.particao.value_counts().to_dict()
    testes[tarefa] = (tabela.loc[tabela.strat_fold.eq(10), cols_id + [alvo]]
                      .sort_values('linha_sinal').reset_index(drop=True))
    teste = testes[tarefa]
    exigir(len(teste) == contagens[tarefa]['teste'] and teste.particao.eq('teste').all(),
           f'{tarefa}: a tabela de teste deve conter exclusivamente todos os ECGs do fold 10.')
    y_treino = tabela.loc[tabela.strat_fold.between(1, 8), alvo]
    constantes_baseline[tarefa] = (int(y_treino.mode().min()) if tarefa == 'classificacao'
                                  else float(y_treino.mean()))
    if tarefa == 'classificacao':
        exigir(set(y_treino) == set(teste[alvo]) == {0, 1},
               'Treino e teste de classificação devem conter ambas as classes esperadas.')
    exigir(np.isfinite(np.asarray(teste[alvo], dtype=float)).all(),
           f'{tarefa}: alvo de teste não finito.')

display(pd.DataFrame(contagens).reindex(['treino', 'validacao', 'teste']))
print('Referências calculadas nos folds 1–8', constantes_baseline)

## 2. Auditoria dos modelos e dos artefatos de treinamento

O código supervisionado aplica `BCEWithLogitsLoss` ou `MSELoss` e seleciona o checkpoint salvo pelo critério de melhora da perda no fold 9. O autoencoder aprende a reconstruir os sinais nos folds 1–8 e seleciona seu checkpoint pela perda de reconstrução no fold 9. Os Random Forests aprendem nos folds 1–8 e têm configurações fixadas na etapa 4. A auditoria verifica as partições e contagens registradas, a seleção do checkpoint no histórico e a assinatura das amostras dos Random Forests. Também compara o hash do checkpoint do autoencoder com o registrado na geração dos embeddings usados pelos modelos clássicos.

As verificações cobrem a procedência registrada pelos notebooks e módulos. Os formatos de treinamento não contêm um registro independente de ações feitas fora desse fluxo. Na regressão, o código existente usa `alvo_idade` diretamente em anos, sem transformação de escala.

In [ ]:
def carregar_estado(caminho):
    try:
        return torch.load(caminho, map_location='cpu', weights_only=True)
    except Exception as exc:
        raise RuntimeError(f'Falha ao carregar pesos em {caminho}: {exc}') from exc


def conferir_config(config, treino, validacao, chave_treino, chave_val, origem):
    exigir(config['hiperparametros']['modo_teste'] is False
           and config['particoes_usadas'] == ['treino', 'validacao']
           and config[chave_treino] == treino and config[chave_val] == validacao,
           f'{origem}: rodada rápida, contagens divergentes ou partições incompatíveis.')


def resumo_historico(caminho, estado, hiper):
    hist = pd.read_csv(caminho)
    colunas = ['perda_treino', 'perda_validacao', 'segundos_epoca', 'segundos_acumulados']
    exigir(0 < len(hist) <= hiper['epocas']
           and np.array_equal(hist.epoca.to_numpy(), np.arange(1, len(hist) + 1))
           and np.isfinite(hist[colunas].to_numpy()).all()
           and (hist.segundos_epoca >= 0).all()
           and np.allclose(hist.segundos_epoca.cumsum(), hist.segundos_acumulados),
           f'{caminho}: histórico inválido.')
    melhor_epoca, melhor_perda = 0, float('inf')
    for linha in hist.itertuples(index=False):
        if linha.perda_validacao < melhor_perda - hiper['delta_minimo']:
            melhor_epoca, melhor_perda = linha.epoca, linha.perda_validacao
    exigir(estado['epoca'] == melhor_epoca
           and np.isclose(estado['perda_validacao'], melhor_perda, rtol=1e-9, atol=1e-12),
           f'{caminho}: checkpoint não corresponde à seleção registrada na validação.')
    exigir(len(hist) == hiper['epocas'] or len(hist) - melhor_epoca >= hiper['paciencia'],
           f'{caminho}: treinamento completo ainda não terminou.')
    melhor_linha = hist.iloc[melhor_epoca - 1]
    return {
        'epocas_executadas': len(hist),
        'melhor_epoca': int(melhor_epoca),
        'perda_treino_na_melhor_epoca': float(melhor_linha.perda_treino),
        'melhor_perda_validacao': float(melhor_perda),
        'segundos_treino_registrados': float(hist.segundos_acumulados.iloc[-1]),
        'historico_origem': str(caminho.relative_to(RAIZ)),
    }


def aplicar_pesos(modelo, estado, caminho):
    try:
        modelo.load_state_dict(estado['modelo'], strict=True)
    except (KeyError, RuntimeError) as exc:
        raise RuntimeError(f'Pesos incompatíveis com a arquitetura existente em {caminho}: {exc}') from exc
    modelo.eval()
    modelo.requires_grad_(False)
    return modelo


redes, florestas, custos, configs_anteriores, metricas_val = {}, {}, {}, {}, {}
for (tarefa, arquitetura), pasta in PASTAS_DL.items():
    config = ler_json(pasta / 'configuracao.json')
    hiper = config['hiperparametros']
    conferir_config(config, contagens[tarefa]['treino'], contagens[tarefa]['validacao'],
                    'amostras_treino', 'amostras_validacao', pasta)
    criterio = 'BCEWithLogitsLoss' if tarefa == 'classificacao' else 'MSELoss'
    entrada = [DIMENSAO_ORIGINAL] if arquitetura == 'mlp' else [12, 1000]
    exigir(hiper['tarefa'] == tarefa and hiper['arquitetura'] == arquitetura
           and config['criterio'] == criterio and config['forma_entrada'] == entrada
           and config['normalizacao'] == normalizacao['metodo'],
           f'{pasta}: tarefa, arquitetura, perda, entrada ou normalização incompatível.')
    caminho = pasta / 'melhor.pt'
    estado = carregar_estado(caminho)
    exigir(estado['configuracao'] == hiper, f'{caminho}: configuração do checkpoint divergente.')
    modelo = criar_modelo(arquitetura, dropout=hiper['dropout'])
    exigir(sum(p.numel() for p in modelo.parameters()) == config['parametros_treinaveis'],
           f'{caminho}: quantidade de parâmetros incompatível.')
    chave = (tarefa, arquitetura)
    redes[chave] = aplicar_pesos(modelo, estado, caminho)
    custos[chave] = {
        **resumo_historico(pasta / 'historico.csv', estado, hiper),
        'dimensao_entrada': DIMENSAO_ORIGINAL,
        'parametros_treinaveis': config['parametros_treinaveis'],
        'dispositivo_treino_registrado': config['dispositivo_inicial'],
    }
    configs_anteriores[chave] = config
    metricas_val[chave] = estado['metricas_validacao']

unicos_ajuste, amostras_rf = selecionar_amostras(tabelas, modo_teste=False, semente=SEMENTE)
config_ae = ler_json(AE_CONFIG)
hiper_ae = config_ae['hiperparametros']
conferir_config(config_ae, int(unicos_ajuste.particao.eq('treino').sum()),
                int(unicos_ajuste.particao.eq('validacao').sum()),
                'ecgs_treino', 'ecgs_validacao', AE_CONFIG)
latente = config_ae['dimensao_latente']
exigir(config_ae['sem_rotulos'] is True and latente == hiper_ae['dimensao_latente']
       and config_ae['entrada'] == [12, 1000]
       and config_ae['dimensao_original'] == DIMENSAO_ORIGINAL,
       f'{AE_CONFIG}: configuração ou dimensões incompatíveis.')
estado_ae = carregar_estado(AE_CHECKPOINT)
exigir(estado_ae['dimensao_latente'] == latente, f'{AE_CHECKPOINT}: dimensão latente incompatível.')
autoencoder = AutoencoderECG1D(latente)
exigir(sum(p.numel() for p in autoencoder.parameters()) == config_ae['parametros_treinaveis'],
       f'{AE_CHECKPOINT}: quantidade de parâmetros incompatível.')
autoencoder = aplicar_pesos(autoencoder, estado_ae, AE_CHECKPOINT)
custo_ae = {
    **resumo_historico(AE_HISTORICO, estado_ae, hiper_ae),
    'parametros_treinaveis': config_ae['parametros_treinaveis'],
    'parametros_encoder': sum(p.numel() for camada in
                              (autoencoder.encoder_convolucional, autoencoder.projecao_latente)
                              for p in camada.parameters()),
    'dimensao_original': DIMENSAO_ORIGINAL, 'dimensao_latente': latente,
    'dispositivo_treino_registrado': config_ae.get('dispositivo_inicial', 'não registrado'),
}
hash_ae = _sha256_arquivo(AE_CHECKPOINT)
meta_ae = ler_json(AE_META)
indice_anterior = pd.read_csv(AE_INDICE)
indice_esperado = unicos_ajuste[cols_id].assign(linha_embedding=np.arange(len(unicos_ajuste)))
z_ajuste = np.load(AE_Z_AJUSTE, mmap_mode='r', allow_pickle=False)
exigir(meta_ae['checkpoint_sha256'] == hash_ae and meta_ae['sem_rotulos_na_entrada'] is True
       and meta_ae['particoes'] == ['treino', 'validacao']
       and meta_ae['dimensao_original'] == DIMENSAO_ORIGINAL and meta_ae['dimensao_latente'] == latente
       and meta_ae['shape'] == list(z_ajuste.shape) == [len(unicos_ajuste), latente]
       and z_ajuste.dtype == np.float32 and indice_anterior.equals(indice_esperado),
       f'{AE_META}: o checkpoint ou o índice não corresponde aos embeddings de treino/validação.')
del z_ajuste

In [ ]:
for tarefa in TAREFAS:
    assinatura = _assinatura_amostra(amostras_rf[tarefa], ALVOS[tarefa])
    hiper_referencia = None
    for rep in REPRESENTACOES:
        caminho, caminho_config, caminho_metricas = RF_ARQUIVOS[tarefa, rep]
        config = ler_json(caminho_config)
        metricas = ler_json(caminho_metricas)
        hiper = config['hiperparametros']
        conferir_config(config, contagens[tarefa]['treino'], contagens[tarefa]['validacao'],
                        'ecgs_treino', 'ecgs_validacao', caminho_config)
        d = DIMENSAO_ORIGINAL if rep == 'original' else latente
        exigir(all(obj['tarefa'] == tarefa and obj['representacao'] == rep
                   and obj['assinatura_amostra'] == assinatura for obj in (config, metricas))
               and metricas['dimensao_entrada'] == d
               and all(metricas[k] == config[k] for k in ('ecgs_treino', 'ecgs_validacao')),
               f'{caminho_config}: métricas ou assinatura do conjunto de ajuste divergentes. '
               'Confira os CSVs preparados e a versão do pandas usada no treinamento.')
        exigir(hiper_referencia is None or hiper == hiper_referencia,
               f'{caminho_config}: hiperparâmetros distintos entre as representações.')
        hiper_referencia = hiper

        try:
            with warnings.catch_warnings():
                warnings.filterwarnings('error', message=r'Trying to unpickle estimator .*',
                                        category=UserWarning)
                rf = joblib.load(caminho)
        except UserWarning as exc:
            versao_origem = getattr(exc, 'original_sklearn_version', None)
            detalhe = f'Instale scikit-learn=={versao_origem}.' if versao_origem else str(exc)
            raise RuntimeError(f'{caminho}: versão de treinamento incompatível. {detalhe} '
                               'Reinicie o ambiente após instalar a versão usada na etapa 4.') from exc
        except Exception as exc:
            raise RuntimeError(f'Falha ao abrir {caminho} com scikit-learn {sklearn.__version__}. '
                               f'Confira a versão usada na etapa 4 e a integridade do arquivo: {exc}') from exc

        tipo = RandomForestClassifier if tarefa == 'classificacao' else RandomForestRegressor
        exigir(isinstance(rf, tipo) and getattr(rf, 'n_features_in_', None) == d
               and len(getattr(rf, 'estimators_', [])) == hiper['n_estimators']
               and all(rf.get_params().get(k) == v for k, v in hiper.items() if k != 'modo_teste'),
               f'{caminho}: tipo, atributos ou configuração do modelo incompatíveis.')
        if tarefa == 'classificacao':
            exigir(np.array_equal(rf.classes_, [0, 1]), f'{caminho}: classes incompatíveis.')
        segundos_rf = metricas['segundos_treinamento']
        exigir(np.isfinite(segundos_rf) and segundos_rf >= 0, f'{caminho_metricas}: tempo inválido.')
        chave = (tarefa, f'rf_{rep}')
        florestas[chave] = rf
        custos[chave] = {
            'dimensao_entrada': d,
            'segundos_treino_registrados': segundos_rf,
            'dispositivo_treino_registrado': 'CPU',
            'arvores': len(rf.estimators_),
            'nos_arvores': sum(arvore.tree_.node_count for arvore in rf.estimators_),
            'threads_rf': rf.n_jobs,
        }
        configs_anteriores[chave] = config
        metricas_val[chave] = metricas

print('Artefatos e protocolo registrados conferidos. Nenhum sinal de teste foi lido até aqui.')

## 3. Previsões e tempos de inferência no fold 10

A MLP e o Random Forest original recebem a ordem vetorizada `(tempo, derivação)`. A CNN, a ResNet e o encoder recebem `(derivação, tempo)`. Todas as redes rodam em `eval()` e `torch.inference_mode()`. Na classificação, as redes usam **logit ≥ 0** (probabilidade ≥ 0,5), como na etapa 3. Os Random Forests usam o `predict` do modelo salvo, como na etapa 4. A regressão usa a saída contínua diretamente em anos.

A inferência é medida em lotes de 64, com um lote de aquecimento fora da medição por modelo ou etapa. Somamos os tempos dos lotes em cada repetição e reportamos a mediana de três totais. O tempo inclui transferências do lote e da saída entre CPU e GPU quando aplicável, com sincronização CUDA. Leitura do Drive ou do binário, montagem do lote em memória, carregamento dos modelos e cálculo de métricas ficam fora da medição. As redes e o encoder usam CUDA quando disponível. Os Random Forests usam CPU com o paralelismo salvo.

O encoder é medido separadamente para cada tarefa, sobre os mesmos ECGs que os demais modelos da tarefa. O custo de inferência do pipeline com embeddings soma a codificação e o `predict` do Random Forest. ECGs elegíveis nas duas tarefas mantêm o mesmo `ecg_id` e `linha_sinal` em cada índice.

In [ ]:
def sincronizar(dispositivo):
    if dispositivo.type == 'cuda':
        torch.cuda.synchronize(dispositivo)


def prever_em_lotes(entradas, prever, dispositivo, n_colunas=None):
    partes = []
    tempos = np.zeros(REPETICOES, dtype=np.float64)
    for i, lote in enumerate(entradas):
        if i == 0:
            prever(lote)
            sincronizar(dispositivo)
        for r in range(REPETICOES):
            inicio = perf_counter()
            saida = prever(lote)
            sincronizar(dispositivo)
            tempos[r] += perf_counter() - inicio
            if r == 0:
                partes.append(saida)
        esperado = (len(lote),) if n_colunas is None else (len(lote), n_colunas)
        exigir(partes[-1].shape == esperado and np.isfinite(partes[-1]).all(),
               f'Saída inválida no lote {i + 1}: esperava {esperado}, recebi {partes[-1].shape}.')
    exigir(bool(partes), 'Nenhum lote disponível para inferência.')
    return np.concatenate(partes), float(np.median(tempos))


def lotes_sinais(sinais, linhas, vetorial):
    for inicio in range(0, len(linhas), LOTE):
        lote = sinais[linhas[inicio:inicio + LOTE]]
        exigir(np.isfinite(lote).all(), f'Sinal não finito no lote iniciado em {inicio}.')
        yield (lote.reshape(len(lote), DIMENSAO_ORIGINAL) if vetorial else
               np.ascontiguousarray(lote.transpose(0, 2, 1)))


@torch.inference_mode()
def executar_torch(funcao, lote):
    entrada = torch.from_numpy(lote).to(DISPOSITIVO)
    return funcao(entrada).cpu().numpy()


def lotes_matriz(matriz):
    for inicio in range(0, len(matriz), LOTE):
        yield matriz[inicio:inicio + LOTE]


def avaliar_fold_10(caminho_binario):
    cpu = torch.device('cpu')
    sinais = np.memmap(caminho_binario, mode='r', dtype=np.float32, shape=shape)
    previsoes, embeddings, tempos = {}, {}, {}
    for tarefa, teste in testes.items():
        linhas = teste.linha_sinal.to_numpy(dtype=np.int64)
        autoencoder.to(DISPOSITIVO)
        try:
            z, segundos_encoder = prever_em_lotes(
                lotes_sinais(sinais, linhas, vetorial=False),
                lambda lote: executar_torch(autoencoder.codificar, lote),
                DISPOSITIVO, n_colunas=latente,
            )
        finally:
            autoencoder.to(cpu)
        exigir(len(z) == len(teste), f'{tarefa}: quantidade de embeddings divergente.')
        indice = teste.copy()
        indice.insert(0, 'linha_embedding', np.arange(len(teste)))
        embeddings[tarefa] = (z, indice)

        for arquitetura in ARQUITETURAS:
            chave = (tarefa, arquitetura)
            modelo = redes[chave].to(DISPOSITIVO)
            try:
                bruto, segundos = prever_em_lotes(
                    lotes_sinais(sinais, linhas, vetorial=arquitetura == 'mlp'),
                    lambda lote: executar_torch(modelo, lote), DISPOSITIVO,
                )
            finally:
                modelo.to(cpu)
            previsoes[chave] = (bruto >= 0).astype(np.int64) if tarefa == 'classificacao' else bruto
            tempos[chave] = (segundos, 0.0, DISPOSITIVO.type.upper())

        for rep in REPRESENTACOES:
            chave = (tarefa, f'rf_{rep}')
            entradas = (lotes_sinais(sinais, linhas, vetorial=True) if rep == 'original'
                        else lotes_matriz(z))
            previsoes[chave], segundos = prever_em_lotes(entradas, florestas[chave].predict, cpu)
            usa_encoder = rep == 'embedding'
            dispositivo_rf = 'CUDA + CPU' if usa_encoder and DISPOSITIVO.type == 'cuda' else 'CPU'
            tempos[chave] = (segundos, segundos_encoder if usa_encoder else 0.0, dispositivo_rf)

        constante = constantes_baseline[tarefa]
        chave = (tarefa, 'baseline')
        previsoes[chave], segundos = prever_em_lotes(
            lotes_matriz(np.empty(len(teste), dtype=np.uint8)),
            lambda lote: np.full(len(lote), constante), cpu,
        )
        tempos[chave] = (segundos, 0.0, 'CPU')
        print(f'{tarefa}: previsões concluídas para {len(teste):,} ECGs do fold 10.')
    return previsoes, embeddings, tempos

In [ ]:
if COPIAR_BINARIO_PARA_CONTENT:
    exigir(shutil.disk_usage('/content').free >= BINARIO.stat().st_size,
           'Espaço insuficiente em /content. Libere espaço ou desative a cópia do binário.')
    with TemporaryDirectory(prefix='ptbxl_avaliacao_', dir='/content') as cache:
        binario_local = Path(cache) / BINARIO.name
        shutil.copyfile(BINARIO, binario_local)
        previsoes, embeddings_teste, tempos = avaliar_fold_10(binario_local)
else:
    previsoes, embeddings_teste, tempos = avaliar_fold_10(BINARIO)

## 4. Tabelas finais e procedência

A precisão e o recall principais tratam **Alterado** como a classe positiva. A tabela também inclui as médias macro de precisão e recall e os números e taxas de falsos positivos e falsos negativos. O resíduo é **idade prevista − idade real**. MAE e RMSE são expressos em anos e MSE em anos².

Os tempos de treino são recuperados dos artefatos. Para as redes e o autoencoder, os históricos incluem o tempo de treino e validação por época. Para os Random Forests, o tempo registrado cobre o ajuste do estimador. O pré-treino do autoencoder entra como custo separado e é compartilhado entre as tarefas. O tempo da extração dos embeddings de ajuste não foi registrado nas etapas anteriores.

In [ ]:
def salvar_csv(tabela, nome, index=False):
    tabela.to_csv(SAIDA / nome, index=index, encoding='utf-8')


def salvar_json(objeto, nome):
    (SAIDA / nome).write_text(json.dumps(objeto, ensure_ascii=False, indent=2, allow_nan=False),
                             encoding='utf-8')


comparacoes, erros_classificacao, grupos_idade = {}, {}, []
tabelas_predicoes, matrizes = {}, {}
for tarefa, teste in testes.items():
    y = teste[ALVOS[tarefa]].to_numpy()
    predicoes_tarefa = teste.copy()
    registros = []
    for modelo, nome in NOMES.items():
        chave = (tarefa, modelo)
        p = previsoes[chave]
        exigir(p.shape == y.shape, f'{tarefa}/{modelo}: quantidade de previsões divergente.')
        predicoes_tarefa[f'previsto_{modelo}'] = p
        segundos_modelo, segundos_encoder, dispositivo = tempos[chave]
        registro = {
            'modelo_id': modelo, 'modelo': nome, 'fold': 10, 'n_teste': len(teste),
            **custos.get(chave, {}),
            'segundos_pre_treino_ae': custo_ae['segundos_treino_registrados'] if modelo == 'rf_embedding' else 0.0,
            'segundos_inferencia_modelo': segundos_modelo,
            'segundos_inferencia_encoder': segundos_encoder,
            'segundos_inferencia_total': segundos_modelo + segundos_encoder,
            'ms_inferencia_por_ecg': 1000 * (segundos_modelo + segundos_encoder) / len(y),
            'dispositivo_inferencia': dispositivo,
        }
        if tarefa == 'classificacao':
            exigir(np.isin(p, [0, 1]).all(), f'{modelo}: previsões fora das classes 0 e 1.')
            matriz = confusion_matrix(y, p, labels=[0, 1])
            matrizes[modelo] = matriz
            vn, fp, fn, vp = (int(v) for v in matriz.ravel())
            registro.update(
                acuracia=accuracy_score(y, p),
                precisao=precision_score(y, p, pos_label=1, zero_division=0),
                recall=recall_score(y, p, pos_label=1, zero_division=0),
                precisao_macro=precision_score(y, p, average='macro', zero_division=0),
                recall_macro=recall_score(y, p, average='macro', zero_division=0),
                f1_macro=f1_score(y, p, average='macro', labels=[0, 1], zero_division=0),
                verdadeiros_negativos=vn, falsos_positivos=fp, falsos_negativos=fn,
                verdadeiros_positivos=vp,
                taxa_falsos_positivos=fp / (vn + fp) if vn + fp else np.nan,
                taxa_falsos_negativos=fn / (fn + vp) if fn + vp else np.nan,
            )
            erros = teste.loc[p != y].copy()
            erros.insert(0, 'modelo', nome)
            erros['classe_prevista'] = p[p != y]
            erros['erro'] = np.where(erros.classe_prevista.eq(1), 'falso_positivo', 'falso_negativo')
            erros_classificacao[modelo] = erros
            chave_metrica = 'f1_macro'
        else:
            residuo = p - y
            predicoes_tarefa[f'residuo_{modelo}_anos'] = residuo
            mse = mean_squared_error(y, p)
            registro.update(mae=mean_absolute_error(y, p), mse=mse, rmse=np.sqrt(mse),
                            r2=r2_score(y, p), residuo_medio_anos=float(residuo.mean()))
            # Faixas fixas para descrever erros, sem mudar o modelo nem o conjunto avaliado.
            faixas = pd.cut(y, [-np.inf, 40, 60, 80, np.inf], right=False,
                            labels=['<40', '40–59', '60–79', '80+'])
            grupos = pd.DataFrame({'faixa_idade': faixas, 'residuo': residuo,
                                   'erro_absoluto': np.abs(residuo)})
            resumo = grupos.groupby('faixa_idade', observed=True).agg(
                n=('residuo', 'size'), residuo_medio_anos=('residuo', 'mean'),
                mae_anos=('erro_absoluto', 'mean'),
            ).reset_index()
            grupos_idade.append(resumo.assign(modelo=nome))
            chave_metrica = 'mae'
        valor_val = metricas_val.get(chave, {}).get(chave_metrica)
        if valor_val is not None:
            exigir(np.isfinite(valor_val), f'{tarefa}/{modelo}: métrica de validação inválida.')
            registro[f'{chave_metrica}_validacao'] = valor_val
            registro[f'delta_{chave_metrica}_teste_menos_validacao'] = registro[chave_metrica] - valor_val
        registros.append(registro)
    comparacoes[tarefa] = pd.DataFrame(registros)
    tabelas_predicoes[tarefa] = predicoes_tarefa

In [ ]:
execucao = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SAIDA = BASE_SAIDA / execucao
SAIDA.mkdir(parents=True, exist_ok=False)
(SAIDA / 'graficos').mkdir()
(SAIDA / 'graficos' / 'treino_validacao').mkdir()
(SAIDA / 'embeddings').mkdir()

salvar_csv(inventario, 'inventario_entradas.csv')
salvar_csv(pd.DataFrame(contagens).rename_axis('particao'), 'contagens_particoes.csv', index=True)
salvar_csv(pd.concat(erros_classificacao.values(), ignore_index=True), 'erros_classificacao.csv')
salvar_csv(pd.concat(grupos_idade, ignore_index=True), 'regressao_por_faixa_idade.csv')

embeddings_resumos = []
for tarefa, tabela in comparacoes.items():
    salvar_csv(tabela, f'comparacao_{tarefa}.csv')
    salvar_csv(tabelas_predicoes[tarefa], f'predicoes_{tarefa}.csv')
    metrica = 'f1_macro' if tarefa == 'classificacao' else 'mae'
    rf_tabela = tabela.loc[tabela.modelo_id.isin(['rf_original', 'rf_embedding'])].copy()
    rf_tabela['tarefa'] = tarefa
    rf_tabela['representacao'] = rf_tabela.modelo_id.map(
        {'rf_original': 'Sinal original vetorizado', 'rf_embedding': 'Embedding do autoencoder'})
    rf_tabela['metrica_principal'] = 'F1 macro' if tarefa == 'classificacao' else 'MAE (anos)'
    rf_tabela['valor'] = rf_tabela[metrica]
    embeddings_resumos.append(rf_tabela[[
        'tarefa', 'representacao', 'dimensao_entrada', 'metrica_principal', 'valor',
        'segundos_treino_registrados', 'segundos_pre_treino_ae', 'segundos_inferencia_modelo',
        'segundos_inferencia_encoder', 'segundos_inferencia_total', 'dispositivo_inferencia',
    ]])
    z, indice = embeddings_teste[tarefa]
    np.save(SAIDA / 'embeddings' / f'{tarefa}_fold10.npy', z, allow_pickle=False)
    salvar_csv(indice, f'embeddings/indice_{tarefa}_fold10.csv')

tabela_embeddings = pd.concat(embeddings_resumos, ignore_index=True)
salvar_csv(tabela_embeddings, 'comparacao_embeddings.csv')
salvar_csv(pd.DataFrame([custo_ae]), 'custo_autoencoder.csv')
salvar_json(
    {'redes_e_florestas': {f'{t}/{m}': c for (t, m), c in configs_anteriores.items()},
     'autoencoder': config_ae},
    'configuracoes_treinamento.json',
)

cpuinfo = Path('/proc/cpuinfo')
nome_cpu = next((l.partition(':')[2].strip()
                 for l in cpuinfo.read_text().splitlines() if l.startswith('model name')),
                platform.processor() or platform.machine()) if cpuinfo.is_file() else platform.machine()
salvar_json({
    'execucao_utc': execucao, 'fold_teste': 10, 'folds_treino': list(range(1, 9)),
    'fold_validacao': 9, 'raiz_projeto': str(RAIZ), 'semente': SEMENTE,
    'baseline_classe': constantes_baseline['classificacao'],
    'baseline_idade_anos': constantes_baseline['regressao'],
    'limiar_logit': 0.0, 'versoes': {'python': platform.python_version(), 'numpy': np.__version__,
        'pandas': pd.__version__, 'torch': str(torch.__version__), 'scikit-learn': sklearn.__version__,
        'matplotlib': matplotlib.__version__, 'joblib': joblib.__version__, 'cuda_torch': torch.version.cuda},
    'cpu': nome_cpu, 'cpus_logicas': os.cpu_count(), 'threads_torch': torch.get_num_threads(),
    'dispositivo_torch': str(DISPOSITIVO),
    'gpu': torch.cuda.get_device_name(DISPOSITIVO) if DISPOSITIVO.type == 'cuda' else None,
    'lote': LOTE, 'repeticoes': REPETICOES, 'lotes_aquecimento': 1,
    'copia_binario_local': COPIAR_BINARIO_PARA_CONTENT,
    'medicao': ('Mediana de três somas dos tempos por lote, incluindo transferências de entrada e saída e '
                'sincronização CUDA quando aplicável. Exclui leitura em disco, preparo dos lotes, '
                'carregamento dos pesos e métricas. RF com embeddings soma medianas do encoder e RF.'),
    'checkpoint_ae_sha256': hash_ae,
    'reducao_dimensional': DIMENSAO_ORIGINAL / latente,
}, 'protocolo_avaliacao.json')

colunas_custo = ['modelo', 'dimensao_entrada', 'parametros_treinaveis', 'epocas_executadas',
                'melhor_epoca', 'segundos_treino_registrados', 'segundos_pre_treino_ae',
                'ms_inferencia_por_ecg', 'dispositivo_inferencia']
for tarefa, metricas in [('classificacao', ['acuracia', 'precisao', 'recall', 'f1_macro']),
                         ('regressao', ['mae', 'mse', 'rmse', 'r2'])]:
    display(comparacoes[tarefa][['modelo', *metricas, 'n_teste', 'ms_inferencia_por_ecg']].round(4))
    display(comparacoes[tarefa][colunas_custo].round(4))
display(tabela_embeddings.round(4))
display(pd.DataFrame([custo_ae]).round(4))

## 5. Gráficos para apresentação

As figuras de avaliação identificam o fold 10. As matrizes mostram contagens e percentuais por classe real. Os gráficos de regressão usam a mesma escala entre modelos para facilitar a comparação. As curvas de aprendizado das redes, a curva de perda do autoencoder e os exemplos de reconstrução são cópias das etapas de origem, com referência em `procedencia_graficos.csv`.

In [ ]:
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
CORES = {'mlp': '#4477AA', 'cnn1d': '#228833', 'resnet1d': '#007C91',
         'rf_original': '#AA3377', 'rf_embedding': '#B85718', 'baseline': '#687681'}


def salvar_figura(fig, nome, exibir=True):
    fig.savefig(SAIDA / 'graficos' / nome, dpi=240, bbox_inches='tight', facecolor='white')
    if exibir:
        plt.show()
    plt.close(fig)


def barras_metrica(tabela, coluna, eixo, titulo, arquivo, limite):
    fig, ax = plt.subplots(figsize=(8.5, 4.6), layout='constrained')
    barras = ax.barh(tabela.modelo, tabela[coluna], color=tabela.modelo_id.map(CORES))
    ax.bar_label(barras, labels=[f'{v:.3f}'.replace('.', ',') for v in tabela[coluna]], padding=6)
    ax.invert_yaxis()
    ax.set(xlabel=eixo, title=titulo, xlim=(0, limite))
    ax.set_axisbelow(True)
    ax.grid(axis='x', alpha=0.2)
    salvar_figura(fig, arquivo)


for modelo, matriz in matrizes.items():
    salvar_csv(pd.DataFrame(matriz, index=pd.Index(CLASSES, name='classe_real'),
                           columns=[f'previsto_{c.lower()}' for c in CLASSES]),
               f'matriz_confusao_{modelo}.csv', index=True)
    totais = matriz.sum(axis=1, keepdims=True)
    proporcao = np.divide(matriz, totais, out=np.zeros_like(matriz, dtype=float), where=totais > 0)
    fig, ax = plt.subplots(figsize=(5.5, 4.5), layout='constrained')
    mapa = ax.imshow(proporcao, cmap='Blues', vmin=0, vmax=1)
    for i, j in np.ndindex(matriz.shape):
        rotulo = f'{matriz[i, j]}\n{proporcao[i, j]:.1%}'
        ax.text(j, i, rotulo, ha='center', va='center',
                color='white' if proporcao[i, j] > 0.52 else '#152832', fontsize=12)
    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=CLASSES, yticklabels=CLASSES,
           xlabel='Classe prevista', ylabel='Classe real',
           title=f'{NOMES[modelo]} · Matriz de confusão\nTeste, fold 10')
    fig.colorbar(mapa, ax=ax, label='Proporção da classe real', shrink=0.85)
    salvar_figura(fig, f'matriz_confusao_{modelo}.png', exibir=False)

barras_metrica(comparacoes['classificacao'], 'f1_macro', 'F1 macro',
               'Classificação de ECG · Teste, fold 10', 'comparacao_f1_macro.png', 1.12)
barras_metrica(comparacoes['regressao'], 'mae', 'Erro absoluto médio (anos)',
               'Estimativa de idade · Teste, fold 10', 'comparacao_mae.png',
               float(comparacoes['regressao'].mae.max() * 1.19))

reais = testes['regressao'].alvo_idade.to_numpy()
todos_previstos = np.concatenate([previsoes['regressao', m] for m in NOMES])
lim_idade = (min(reais.min(), todos_previstos.min()) - 2, max(reais.max(), todos_previstos.max()) + 2)
lim_residuo = max(np.max(np.abs(previsoes['regressao', m] - reais)) for m in NOMES) + 2
for residuo, arquivo, titulo, eixo_y in [
    (False, 'idade_real_vs_prevista.png', 'Idade real e prevista', 'Idade prevista (anos)'),
    (True, 'residuos_regressao.png', 'Resíduos por idade real', 'Prevista − real (anos)'),
]:
    fig, axes = plt.subplots(2, 3, figsize=(13.5, 8), sharex=True, sharey=True, layout='constrained')
    for ax, (modelo, nome) in zip(axes.flat, NOMES.items()):
        p = previsoes['regressao', modelo]
        if residuo:
            ax.axhline(0, ls='--', lw=1, c='#343A40')
            ax.set_ylim(-lim_residuo, lim_residuo)
        else:
            ax.plot(lim_idade, lim_idade, ls='--', lw=1, c='#343A40')
            ax.set_ylim(lim_idade)
        ax.scatter(reais, p - reais if residuo else p, s=9, alpha=0.22,
                   color=CORES[modelo], edgecolors='none')
        ax.set(title=nome, xlim=lim_idade)
        ax.grid(alpha=0.14)
    fig.supxlabel('Idade real (anos)')
    fig.supylabel(eixo_y)
    fig.suptitle(f'{titulo} · Teste, fold 10', fontsize=16)
    salvar_figura(fig, arquivo)

In [ ]:
graficos_origem = [
    (pasta / 'curvas.png', f'curvas_{tarefa}_{arquitetura}.png', 'Treino e validação')
    for (tarefa, arquitetura), pasta in PASTAS_DL.items()
]
graficos_origem += [
    (GRAFICOS_AE[0], 'perda_reconstrucao.png', 'Treino e validação'),
    (GRAFICOS_AE[1], 'exemplos_reconstrucao.png', 'Exemplos de treino'),
]
procedencia = []
for origem, nome, particoes in graficos_origem:
    destino = SAIDA / 'graficos' / 'treino_validacao' / nome
    shutil.copyfile(origem, destino)
    procedencia.append({'origem': str(origem.relative_to(RAIZ)),
                        'copia': str(destino.relative_to(SAIDA)), 'particoes': particoes})
salvar_csv(pd.DataFrame(procedencia), 'procedencia_graficos.csv')

display(Markdown('**Autoencoder · curva de treino e validação**'))
display(Image(filename=str(SAIDA / 'graficos' / 'treino_validacao' / 'perda_reconstrucao.png')))
display(Markdown('**Autoencoder · exemplos do conjunto de treino**'))
display(Image(filename=str(SAIDA / 'graficos' / 'treino_validacao' / 'exemplos_reconstrucao.png')))

## 6. Discussão científica

A discussão é gerada com as medições da execução. As diferenças entre partições são descritivas e devem ser lidas junto com as curvas de treino e validação. Os resultados descrevem generalização para pacientes reservados do PTB-XL. A extrapolação para outras populações ou equipamentos exigiria validação externa.

In [ ]:
def numero(valor, casas=3):
    return f'{valor:.{casas}f}'.replace('.', ',')


cl = comparacoes['classificacao'].set_index('modelo_id')
rg = comparacoes['regressao'].set_index('modelo_id')
mcl = cl.drop(index='baseline').sort_values('f1_macro', ascending=False).iloc[0]
mrg = rg.drop(index='baseline').sort_values('mae').iloc[0]
rapido_cl = cl.drop(index='baseline').sort_values('ms_inferencia_por_ecg').iloc[0]
rapido_rg = rg.drop(index='baseline').sort_values('ms_inferencia_por_ecg').iloc[0]
falsos_cl = cl.drop(index='baseline')
menos_fp = falsos_cl.loc[falsos_cl.falsos_positivos.eq(falsos_cl.falsos_positivos.min()), 'modelo']
menos_fn = falsos_cl.loc[falsos_cl.falsos_negativos.eq(falsos_cl.falsos_negativos.min()), 'modelo']
delta_f1 = cl.loc['rf_embedding', 'f1_macro'] - cl.loc['rf_original', 'f1_macro']
delta_mae = rg.loc['rf_embedding', 'mae'] - rg.loc['rf_original', 'mae']
delta_latencia_cl = cl.loc['rf_embedding', 'ms_inferencia_por_ecg'] - cl.loc['rf_original', 'ms_inferencia_por_ecg']
delta_latencia_rg = rg.loc['rf_embedding', 'ms_inferencia_por_ecg'] - rg.loc['rf_original', 'ms_inferencia_por_ecg']


def diferenca(valor, mais, menos, empate):
    if np.isclose(valor, 0, rtol=0, atol=1e-12):
        return empate
    return (mais if valor > 0 else menos).format(numero(abs(valor)))


efeito_cl = diferenca(delta_f1, 'elevou o F1 macro em {}', 'reduziu o F1 macro em {}',
                      'manteve o F1 macro')
efeito_rg = diferenca(delta_mae, 'aumentou o MAE em {} anos', 'diminuiu o MAE em {} anos',
                      'manteve o MAE')
efeito_tempo_cl = diferenca(delta_latencia_cl, 'aumentou em {} ms por ECG',
                           'diminuiu em {} ms por ECG', 'foi igual')
efeito_tempo_rg = diferenca(delta_latencia_rg, 'aumentou em {} ms por ECG',
                           'diminuiu em {} ms por ECG', 'foi igual')
classe_baseline = CLASSES[constantes_baseline['classificacao']]
ganho_cl = mcl.f1_macro - cl.loc['baseline', 'f1_macro']
ganho_rg = rg.loc['baseline', 'mae'] - mrg.mae
ref_cl = diferenca(ganho_cl, 'superou a referência em {} pontos de F1 macro',
                   'ficou {} pontos de F1 macro abaixo da referência', 'empatou com a referência em F1 macro')
ref_rg = diferenca(ganho_rg, 'reduziu o MAE da referência em {} anos',
                   'teve MAE {} anos maior que o da referência', 'empatou com a referência em MAE')
vies = diferenca(mrg.residuo_medio_anos, 'superestimou a idade em {} anos, em média',
                  'subestimou a idade em {} anos, em média', 'teve resíduo médio igual a zero')

faixas_mrg = pd.concat(grupos_idade, ignore_index=True)
faixa_erro = faixas_mrg.loc[faixas_mrg.modelo.eq(mrg.modelo)].sort_values('mae_anos').iloc[-1]

linhas = [
    '### Classificação',
    f"{mcl.modelo} obteve o maior F1 macro entre as cinco abordagens ({numero(mcl.f1_macro)}) e "
    f"{ref_cl}. A classe majoritária de treino foi **{classe_baseline}**. "
    f"Para {mcl.modelo}, houve {int(mcl.falsos_positivos)} ECGs normais previstos como alterados "
    f"({numero(100 * mcl.taxa_falsos_positivos, 1)}% dos normais) e "
    f"{int(mcl.falsos_negativos)} ECGs alterados previstos como normais "
    f"({numero(100 * mcl.taxa_falsos_negativos, 1)}% dos alterados). "
    f"A menor contagem de falsos positivos foi {int(falsos_cl.falsos_positivos.min())} "
    f"({', '.join(menos_fp)}). A menor contagem de falsos negativos foi "
    f"{int(falsos_cl.falsos_negativos.min())} ({', '.join(menos_fn)}). "
    "Cada contagem deve ser interpretada com a taxa e o F1 macro, pois a frequência das classes afeta a acurácia.",
    '### Regressão',
    f"{mrg.modelo} teve o menor MAE entre as cinco abordagens ({numero(mrg.mae)} anos) e {ref_rg}. "
    f"A referência previu {numero(constantes_baseline['regressao'], 2)} anos para todos os exames. "
    f"Com R² de {numero(mrg.r2)}, {mrg.modelo} {vies}. "
    "O resíduo médio resume a direção do erro e pode ocultar erros de sinais opostos. "
    f"O maior MAE de {mrg.modelo} por faixa ocorreu em {faixa_erro.faixa_idade} anos "
    f"({numero(faixa_erro.mae_anos)} anos de MAE, {int(faixa_erro.n)} exames nessa faixa).",
    '### Informação nos embeddings',
    f"O encoder reduz {DIMENSAO_ORIGINAL} atributos a {latente} ({numero(DIMENSAO_ORIGINAL / latente, 1)} vezes). "
    f"Com o mesmo Random Forest e os mesmos exames, usar o embedding {efeito_cl} e {efeito_rg}. "
    "Todos os valores por representação estão na tabela de embeddings. A diferença mede a utilidade "
    "da representação para esta configuração de Random Forest em cada tarefa. "
    "A curva de reconstrução compara treino e validação. Os exemplos de traçados vêm do treino. "
    "A utilidade para os alvos é avaliada pelas previsões no fold 10.",
    '### Custo computacional',
    f"{rapido_cl.modelo} teve o menor tempo por ECG entre as cinco abordagens na classificação "
    f"({numero(rapido_cl.ms_inferencia_por_ecg)} ms por ECG, {rapido_cl.dispositivo_inferencia}, "
    f"F1 macro {numero(rapido_cl.f1_macro)}). Na regressão foi {rapido_rg.modelo} "
    f"({numero(rapido_rg.ms_inferencia_por_ecg)} ms por ECG, {rapido_rg.dispositivo_inferencia}, "
    f"MAE {numero(rapido_rg.mae)} anos). Os melhores em F1 macro e MAE levaram, respectivamente, "
    f"{numero(mcl.ms_inferencia_por_ecg)} e {numero(mrg.ms_inferencia_por_ecg)} ms por ECG. "
    f"Esses tempos descrevem lotes de até {LOTE} exames no ambiente registrado "
    "e não informam a latência de um exame isolado.",
    f"O pré-treino do autoencoder levou {numero(custo_ae['segundos_treino_registrados'], 1)} s "
    f"em {custo_ae['epocas_executadas']} épocas, com checkpoint selecionado na "
    f"época {custo_ae['melhor_epoca']}. Esse custo foi compartilhado pelas duas tarefas. "
    f"Somando encoder e Random Forest, a inferência em relação à entrada original {efeito_tempo_cl} "
    f"na classificação e {efeito_tempo_rg} na regressão. A tabela de custo distingue "
    "parâmetros treináveis das redes, nós das florestas e os componentes de tempo registrados.",
    "As redes com maior e menor contagem de parâmetros na classificação foram "
    f"{cl.loc[list(ARQUITETURAS)].sort_values('parametros_treinaveis').iloc[-1].modelo} e "
    f"{cl.loc[list(ARQUITETURAS)].sort_values('parametros_treinaveis').iloc[0].modelo}. "
    "O número de parâmetros informa o tamanho da rede. A latência também depende "
    "das operações e do dispositivo, conforme os tempos medidos.",
    '### Generalização',
]

for tarefa, tabela, coluna in [('classificação', cl, 'f1_macro'), ('regressão', rg, 'mae')]:
    col_delta = f'delta_{coluna}_teste_menos_validacao'
    medidos = tabela.drop(index='baseline')
    variacoes = ', '.join(f'{linha.modelo} ({numero(linha[col_delta])})'
                          for _, linha in medidos.iterrows())
    nome_metrica = 'F1 macro' if coluna == 'f1_macro' else 'MAE em anos'
    linhas.append(f"Na {tarefa}, as diferenças teste menos validação em {nome_metrica} "
                  f"foram {variacoes}.")
linhas.append(
    "Leia as diferenças junto das curvas para examinar sobreajuste. Uma perda de treino "
    "decrescente com deterioração persistente na validação sugere sobreajuste. Desempenho "
    "baixo nos dois conjuntos pode indicar subajuste, dependendo da curva e da referência. "
    "As diferenças no fold 10 também refletem a composição dos conjuntos. A avaliação usa "
    "um único fold de teste, sem estimativas de incerteza, e compara configurações fixas."
)

discussao = '\n\n'.join(linhas)
(SAIDA / 'discussao.md').write_text(discussao + '\n', encoding='utf-8')
display(Markdown(discussao))
print(f'Resultados persistidos em {SAIDA}')